In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm


# Config of Model Training

In [ ]:
EPOCHS = 100
# BATCH_SIZE = 128
LR = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
print(f"Using device: {DEVICE}")


# Loading in training and test datasets for CIFAR 100

In [ ]:
# Data Loaders
cifar100_mean, cifar100_std = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761) # for CIFAR100 dataset
DATA_TRANSFORM = transforms.Compose([
                          transforms.RandomCrop(32, padding=4),
                          transforms.RandomHorizontalFlip(),
                          transforms.ToTensor(),
                          transforms.Normalize(cifar100_mean, cifar100_std)])

full_data = datasets.CIFAR100(root='./data', train=True, download=True, transform=DATA_TRANSFORM)

train_split = int(0.8*len(full_data))
val_split = len(full_data) - train_split

train_data, val_data = torch.utils.data.random_split(full_data, [train_split, val_split],
                                                     generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_data,   batch_size=256, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

# Import ResNet18 Model

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False) # adapt for 32x32 format of CIFAR100
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Training Metrics and Utilities

In [ ]:
@torch.no_grad()
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    """Compute classification accuracy from logits."""
    return (logits.argmax(1) == y).float().mean().item()

scaler = torch.amp.GradScaler(DEVICE)
def train_one_epoch(model: nn.Module, loader: DataLoader, criterion, optimizer) -> tuple[float, float]:
    """Standard training loop for one epoch."""
    model.train()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:                      # x: [B, C, H, W]
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(DEVICE):
          logits = model(x)                    # [B, 10]
          loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        bs = x.size(0)
        tot += bs
        tot_loss   += loss.item() * bs
        tot_correct+= (logits.argmax(1) == y).sum().item()
    return tot_loss / tot, tot_correct / tot


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion) -> tuple[float, float]:
    """Evaluation loop."""
    model.eval()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        bs = x.size(0)
        tot += bs
        tot_loss   += loss.item() * bs
        tot_correct+= (logits.argmax(1) == y).sum().item()
    return tot_loss / tot, tot_correct / tot

def plot_history(history: dict, title: str):
    """Plot training history."""
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'],   label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history['train_accs'], label='Train Acc')
    plt.plot(history['val_accs'],   label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend()
    plt.suptitle(title)
    plt.tight_layout(); plt.show()


@torch.no_grad()
def show_errors(model: nn.Module, loader: DataLoader, n: int = 6):
    """Display misclassified examples."""
    model.eval(); shown = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(1)
        wrong = torch.where(preds != y)[0]
        for i in wrong[:n]:
            # Check if the image is 1-channel or 3-channel
            img_data = x[i].cpu().squeeze()
            if img_data.ndim == 2: # Grayscale image
                plt.imshow(img_data, cmap='gray')
            else: # Color image (e.g., from transfer learning transform)
                # Permute dimensions from (C, H, W) to (H, W, C) for matplotlib
                plt.imshow(img_data.permute(1, 2, 0))

            # plt.title(f"True: {CLASSES[y[i].item()]} | Pred: {CLASSES[preds[i].item()]}")
            plt.axis('off'); plt.show(); shown += 1
        if shown >= n: break

# Training Run

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_accs": [], "val_accs": []}
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accs"].append(train_acc)
    history["val_accs"].append(val_acc)
    print(f"  Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.3f} | Val: {val_acc:.3f}")


torch.save(model.state_dict(), "resnet18_cifar100.pth")
print("\nDone. Model saved to resnet18_cifar100.pth")

In [ ]:
plot_history(history, "Training and Validation Metrics Over Epochs")

## Ablation Studies


In [ ]:
import copy
import itertools
import os
import time
from typing import Dict, List, Tuple
from torch.utils.tensorboard import SummaryWriter
LOG_DIR = "runs"

Data helper

In [ ]:
def get_augmentation_transform(strategy) -> transforms.Compose:

    normalize = transforms.Normalize(cifar100_mean, cifar100_std)

    if strategy == "none":
        return transforms.Compose([transforms.ToTensor(), normalize])

    elif strategy == "standard":
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
        ])

    elif strategy == "strong":
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                   saturation=0.4, hue=0.1),
            transforms.ToTensor(),
            normalize,
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
        ])

    elif strategy == "cutout":
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
            transforms.RandomErasing(p=1.0, scale=(0.0625, 0.25),
                                     ratio=(1.0, 1.0)),
        ])


def build_loaders(aug_strategy: str = "standard",
                  batch_size: int = 128) -> Tuple[DataLoader, DataLoader]:
    train_ds = datasets.CIFAR100(root="./data", train=True, download=True,
                                 transform=get_augmentation_transform(aug_strategy))
    val_ds   = datasets.CIFAR100(root="./data", train=True, download=True,
                                 transform=DATA_TRANSFORM)

    train_split = int(0.8*len(full_data))
    val_split = len(full_data) - train_split

    train_data, val_data = torch.utils.data.random_split(full_data, [train_split, val_split],
                                                        generator=torch.Generator().manual_seed(SEED))
    train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=128, shuffle=False)

    return train_loader, val_loader

Architectural Variants

In [ ]:
def build_arch_variant(variant: str) -> nn.Module:

    if variant == "resnet18_baseline":
        return model

    elif variant == "resnet18_no_pretrain":
        m = models.resnet18(weights=None)
        m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        m.fc      = nn.Linear(m.fc.in_features, 100)
        return m

    elif variant == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        m.fc      = nn.Linear(m.fc.in_features, 100)
        return m

    elif variant == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        m.fc      = nn.Linear(m.fc.in_features, 100)
        return m

    elif variant == "shallow_resnet":
        m = model
        m.layer4 = nn.Identity()
        m.fc = nn.Linear(256, 100)
        return m

ARCH_VARIANTS = [
    "resnet18_baseline",
    "resnet18_no_pretrain",
    "resnet34",
    "resnet50",
    "shallow_resnet",
]

# Regularization Variants

In [ ]:
class ResNet18WithDropout(nn.Module):
    def __init__(self, dropout_p: float = 0.5):
        super().__init__()
        base = model

        self.backbone = nn.Sequential(
            base.conv1, base.bn1, base.relu,
            base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4,
            base.avgpool,
        )
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc      = nn.Linear(512, 100)

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)


REGULARIZATION_CONFIGS: List[Dict] = [
    {"name": "no_reg",          "wd": 0.0,    "dropout": 0.0,  "aug": "none"},
    {"name": "wd_only",         "wd": 5e-4,   "dropout": 0.0,  "aug": "none"},
    {"name": "aug_only",        "wd": 0.0,    "dropout": 0.0,  "aug": "standard"},
    {"name": "dropout_only",    "wd": 0.0,    "dropout": 0.5,  "aug": "none"},
    {"name": "wd_aug",          "wd": 5e-4,   "dropout": 0.0,  "aug": "standard"},
    {"name": "wd_dropout",      "wd": 5e-4,   "dropout": 0.5,  "aug": "none"},
    {"name": "aug_dropout",     "wd": 0.0,    "dropout": 0.5,  "aug": "strong"},
    {"name": "all_reg",         "wd": 5e-4,   "dropout": 0.5,  "aug": "strong"},
    {"name": "strong_aug_only", "wd": 0.0,    "dropout": 0.0,  "aug": "strong"},
    {"name": "cutout_aug",      "wd": 5e-4,   "dropout": 0.0,  "aug": "cutout"},
]

TTA Variants

In [ ]:
TTA_TRANSFORMS = [
    transforms.Compose([transforms.ToTensor(),
                        transforms.Normalize(cifar100_mean, cifar100_std)]),
    transforms.Compose([transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(),
                        transforms.Normalize(cifar100_mean, cifar100_std)]),
    transforms.Compose([transforms.RandomCrop(32, padding=4),
                        transforms.ToTensor(),
                        transforms.Normalize(cifar100_mean, cifar100_std)]),
    transforms.Compose([transforms.RandomCrop(32, padding=4),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(),
                        transforms.Normalize(cifar100_mean, cifar100_std)]),
    transforms.Compose([transforms.ColorJitter(brightness=0.2, contrast=0.2),
                        transforms.ToTensor(),
                        transforms.Normalize(cifar100_mean, cifar100_std)]),
]

@torch.no_grad()
def tta_evaluate(
    model: nn.Module,
    val_dataset,
    indices: List[int],
    device: torch.device,
    strategy: str = "prob_avg",
    n_augments: int = 5,
) -> float:

    model.eval()
    correct, total = 0, 0
    tta_tfms = TTA_TRANSFORMS[:n_augments]

    for idx in indices:
        raw_img, label = val_dataset[idx]

        if strategy == "single":
            img = DATA_TRANSFORM(raw_img).unsqueeze(0).to(device)
            pred = model(img).argmax(1).item()

        elif strategy in ("prob_avg", "logit_avg"):
            preds = []
            for tfm in tta_tfms:
                img = tfm(raw_img).unsqueeze(0).to(device)
                out = model(img)
                preds.append(torch.softmax(out, 1) if strategy == "prob_avg" else out)
            pred = torch.stack(preds, 0).mean(0).argmax(1).item()

        elif strategy == "majority":
            votes = []
            for tfm in tta_tfms:
                img = tfm(raw_img).unsqueeze(0).to(device)
                votes.append(model(img).argmax(1).item())
            pred = max(set(votes), key=votes.count)

        else:
            raise ValueError(f"Unknown TTA strategy: {strategy!r}")

        correct += int(pred == label)
        total   += 1

    return correct / total

In [ ]:
def run_experiment(
    exp_name: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    weight_decay: float = 5e-4,
    epochs: int = EPOCHS,
    device: torch.device = DEVICE,
) -> Dict:

    writer = SummaryWriter(log_dir=os.path.join(LOG_DIR, exp_name))
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0
    history = {"train_loss": [], "val_loss": [], "train_accs": [], "val_accs": []}
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accs"].append(train_acc)
        history["val_accs"].append(val_acc)
        print(f"  Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.3f} | Val: {val_acc:.3f}")

        writer.add_scalars("Loss",     {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("Accuracy", {"train": train_acc,  "val": val_acc},  epoch)
        writer.add_scalar("LR", scheduler.get_last_lr()[0], epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = copy.deepcopy(model.state_dict())


    writer.close()
    return {
        "exp_name":    exp_name,
        "best_val_acc": best_val_acc,
        "history":     history,
        "best_state":  best_state,
    }

Runners

In [ ]:
def run_architecture_ablations(epochs: int = EPOCHS) -> List[Dict]:

    train_loader, val_loader = build_loaders("standard")
    results = []
    for variant in ARCH_VARIANTS:
        print(f"\n{'='*60}\n  ARCH ABLATION: {variant}\n{'='*60}")
        model = build_arch_variant(variant)
        res = run_experiment(
            exp_name = f"arch/{variant}",
            model = model,
            train_loader= train_loader,
            val_loader = val_loader,
            weight_decay= 5e-4,
            epochs = epochs,
        )
        results.append(res)
    return results


def run_regularization_ablations(epochs: int = EPOCHS) -> List[Dict]:

    results = []
    for cfg in REGULARIZATION_CONFIGS:
        name = cfg["name"]
        wd = cfg["wd"]
        dropout = cfg["dropout"]
        aug = cfg["aug"]

        print(f"\n{'='*60}\n  REG ABLATION: {name}\n{'='*60}")

        train_loader, val_loader = build_loaders(aug)
        model = (ResNet18WithDropout(dropout_p=dropout)
                 if dropout > 0.0 else model)

        res = run_experiment(
            exp_name = f"reg/{name}",
            model = model,
            train_loader = train_loader,
            val_loader = val_loader,
            weight_decay = wd,
            epochs = epochs,
        )
        res["config"] = cfg
        results.append(res)
    return results



def run_tta_ablations(
    trained_model: nn.Module,
    n_eval_samples: int = 2000,
) -> Dict[str, float]:

    raw_dataset = datasets.CIFAR100(root="./data", train=True, download=True,
                                    transform=None)
    n_train = int(0.8 * len(raw_dataset))
    gen     = torch.Generator().manual_seed(SEED)
    _, val_idx = torch.utils.data.random_split(
        range(len(raw_dataset)), [n_train, len(raw_dataset) - n_train], generator=gen
    )
    val_indices = list(val_idx)[:n_eval_samples]

    writer   = SummaryWriter(log_dir=os.path.join(LOG_DIR, "tta"))
    tta_strats = ["single", "prob_avg", "logit_avg", "majority"]
    tta_results = {}

    for n_aug in [1, 3, 5]:
        for strat in tta_strats:
            if strat == "single" and n_aug > 1:
                continue

            key = f"{strat}_n{n_aug}" if strat != "single" else "single_view"
            print(f"  TTA: strategy={strat!r}  n_aug={n_aug}  ...", end=" ", flush=True)
            acc = tta_evaluate(trained_model, raw_dataset, val_indices,
                               DEVICE, strategy=strat, n_augments=n_aug)
            tta_results[key] = acc
            writer.add_scalar("TTA_Accuracy", acc,
                               global_step=tta_strats.index(strat) * 3 + n_aug)
            print(f"acc={acc:.4f}")

    writer.close()
    return tta_results

def print_results_table(results: List[Dict], title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  {'Experiment':<35} {'Best Val Acc':>12}")
    print(f"  {'-'*35} {'-'*12}")
    sorted_res = sorted(results, key=lambda r: r["best_val_acc"], reverse=True)
    for r in sorted_res:
        print(f"  {r['exp_name']:<35} {r['best_val_acc']:>12.4f}")
    print()


def print_tta_table(tta_results: Dict[str, float]):
    print(f"\n{'='*60}")
    print("  Inference-Time Aggregation Results")
    print(f"{'='*60}")
    print(f"  {'Strategy':<30} {'Accuracy':>10}")
    print(f"  {'-'*30} {'-'*10}")
    for k, v in sorted(tta_results.items(), key=lambda x: -x[1]):
        print(f"  {k:<30} {v:>10.4f}")
    print()

Results

In [ ]:
arch_results = run_architecture_ablations(epochs=EPOCHS)
print_results_table(arch_results, "Architecture Ablations")

In [ ]:
reg_results = run_regularization_ablations(epochs=EPOCHS/2)
print_results_table(reg_results, "Regularisation Ablations")

In [ ]:
best = max(arch_results, key=lambda r: r["best_val_acc"])
print(f"\nUsing best arch model: {best['exp_name']} "
      f"(val_acc={best['best_val_acc']:.4f})")

In [ ]:
tta_model = build_arch_variant(
    best["exp_name"].replace("arch/", ""))
tta_model.load_state_dict(best["best_state"])
tta_model = tta_model.to(DEVICE)
tta_results = run_tta_ablations(tta_model, n_eval_samples=2000)
print_tta_table(tta_results)

# HD-CNN Training (in progress)

In [ ]:
# class SharedBackbone(nn.Module):
#     """ResNet18 stem through layer2 — shared by all components."""
#     def __init__(self, pretrained=True):
#         super().__init__()
#         bb = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#         self.stem = nn.Sequential(
#             bb.conv1, bb.bn1, bb.relu, bb.maxpool,
#             bb.layer1, bb.layer2
#         )

#     def forward(self, x):
#         return self.stem(x)


# class CoarseComponent(nn.Module):
#     """
#     Coarse component B:
#       independent layers: layer3, layer4, GAP, FC(512→num_fine)
#       + fine-to-coarse aggregation
#     """
#     def __init__(self, coarse_groups, num_fine=100, pretrained=True):
#         super().__init__()
#         self.coarse_groups = coarse_groups
#         self.K = len(coarse_groups)
#         self.num_fine = num_fine

#         bb = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#         self.backend = nn.Sequential(bb.layer3, bb.layer4,
#                                       nn.AdaptiveAvgPool2d(1))
#         self.fc = nn.Linear(512, num_fine)

#         # Register group membership as buffer for GPU transfers
#         # coarse_mask[k, j] = 1 if fine class j is in coarse group k
#         mask = torch.zeros(self.K, num_fine)
#         for k, group in enumerate(coarse_groups):
#             for j in group:
#                 mask[k, j] = 1.0
#         self.register_buffer('coarse_mask', mask)

#     def forward(self, shared_feat):
#         feat = self.backend(shared_feat).flatten(1)   # (B, 512)
#         fine_logits = self.fc(feat)                    # (B, 100)
#         fine_probs  = torch.softmax(fine_logits, dim=1)

#         # Aggregate: coarse_probs[k] = sum of fine probs in group k
#         # mask: (K, 100) → broadcast over batch
#         coarse_probs = (fine_probs.unsqueeze(1) *
#                         self.coarse_mask.unsqueeze(0)).sum(dim=2)  # (B, K)

#         # L1 normalise (handles overlapping groups)
#         coarse_probs = coarse_probs / coarse_probs.sum(
#             dim=1, keepdim=True).clamp(min=1e-8)

#         return fine_logits, fine_probs, coarse_probs


# class FineComponent(nn.Module):
#     """
#     One fine component F_k:
#       independent layers: layer3, layer4, GAP, FC(512→|group|)
#     """
#     def __init__(self, group_size, pretrained=False):
#         super().__init__()
#         bb = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#         self.backend = nn.Sequential(bb.layer3, bb.layer4,
#                                       nn.AdaptiveAvgPool2d(1))
#         self.fc = nn.Linear(512, group_size)

#     def forward(self, shared_feat):
#         feat = self.backend(shared_feat).flatten(1)
#         return self.fc(feat)   # (B, |group|)


# class HDCNN(nn.Module):
#     """
#     Full HD-CNN:
#       shared → coarse component B + K fine components {F_k}
#              → probabilistic averaging layer
#     """
#     def __init__(self, coarse_groups, num_fine=100, pretrained_backbone=True):
#         super().__init__()
#         self.coarse_groups = coarse_groups
#         self.K = len(coarse_groups)
#         self.num_fine = num_fine

#         self.shared = SharedBackbone(pretrained=pretrained_backbone)
#         self.coarse = CoarseComponent(coarse_groups, num_fine,
#                                         pretrained=pretrained_backbone)
#         self.fine_components = nn.ModuleList([
#             FineComponent(len(g), pretrained=False)
#             for g in coarse_groups
#         ])

#         # Scatter index buffers: fine_indices[k][j] = global class index
#         # Stored as list of tensors; moved to device via .to()
#         self._group_tensors = [torch.tensor(g, dtype=torch.long)
#                                for g in coarse_groups]

#     def to(self, device, **kwargs):
#         super().to(device, **kwargs)
#         self._group_tensors = [t.to(device) for t in self._group_tensors]
#         return self

#     def forward(self, x, active_k=None):
#         """
#         active_k: list of coarse group indices to evaluate (None = all).
#                   Set at inference for conditional execution.
#         Returns:
#             final_probs  (B, 100)
#             coarse_probs (B, K)
#             fine_logits_B (B, 100)   ← coarse component's fine-level logits
#         """
#         shared_feat = self.shared(x)

#         fine_logits_B, _, coarse_probs = self.coarse(shared_feat)

#         if active_k is None:
#             active_k = list(range(self.K))

#         # Weighted scatter accumulation
#         final_probs = torch.zeros(x.size(0), self.num_fine,
#                                    device=x.device, dtype=torch.float32)
#         weight_sum  = torch.zeros(x.size(0), 1,
#                                    device=x.device, dtype=torch.float32)

#         for k in active_k:
#             w_k = coarse_probs[:, k].unsqueeze(1)              # (B, 1)
#             logits_k = self.fine_components[k](shared_feat)    # (B, |g_k|)
#             probs_k  = torch.softmax(logits_k, dim=1)          # (B, |g_k|)

#             # Scatter into full 100-dim space
#             idx = self._group_tensors[k]                       # (|g_k|,)
#             final_probs.scatter_add_(
#                 1,
#                 idx.unsqueeze(0).expand(x.size(0), -1),
#                 w_k * probs_k
#             )
#             weight_sum += w_k

#         final_probs = final_probs / weight_sum.clamp(min=1e-8)
#         return final_probs, coarse_probs, fine_logits_B

In [ ]:
# class HDCNNLoss(nn.Module):
#     """
#     E = NLL(final_probs, targets)
#       + λ * Σ_k (t_k - mean_batch(B_ik))²

#     t_k = fraction of *all* training images belonging to coarse group k.
#     """
#     def __init__(self, coarse_groups, num_fine=100, lam=20.0):
#         super().__init__()
#         self.lam = lam
#         self.K   = len(coarse_groups)

#         # Compute t_k assuming uniform class distribution
#         t = torch.zeros(self.K)
#         total = sum(len(g) for g in coarse_groups)   # may be > 100 if overlapping
#         for k, g in enumerate(coarse_groups):
#             t[k] = len(g) / total
#         self.register_buffer('t_k', t)

#     def forward(self, final_probs, coarse_probs, targets):
#         # NLL on final probabilistic prediction
#         nll = -torch.log(final_probs[range(len(targets)), targets]
#                          .clamp(min=1e-8)).mean()

#         # Coarse category consistency regulariser
#         batch_coarse_mean = coarse_probs.mean(dim=0)          # (K,)
#         consistency = self.lam * ((self.t_k - batch_coarse_mean) ** 2).sum()

#         return nll + consistency, nll.item(), consistency.item()

In [ ]:
# from sklearn.cluster import SpectralClustering

# @torch.no_grad()
# def build_confusion_matrix(model, loader, num_classes=100):
#     model.eval()
#     counts = np.zeros((num_classes, num_classes), dtype=np.float64)
#     for x, y in loader:
#         x = x.to(DEVICE)
#         preds = model(x).argmax(dim=1).cpu().numpy()
#         for t, p in zip(y.numpy(), preds):
#             counts[t, p] += 1
#     row_sums = counts.sum(axis=1, keepdims=True).clip(min=1)
#     return counts / row_sums

# def build_hierarchy(conf_mat, K=9, gamma=5):
#     D = 1.0 - conf_mat
#     np.fill_diagonal(D, 0.0)
#     D = 0.5 * (D + D.T)
#     affinity = np.exp(-D)

#     sc = SpectralClustering(n_clusters=K, affinity='precomputed', random_state=42, n_init=10)
#     labels = sc.fit_predict(affinity)

#     disjoint = [[] for _ in range(K)]
#     for cls, k in enumerate(labels):
#         disjoint[k].append(cls)

#     # Overlapping extension
#     u_t = 1.0 / (gamma * K)
#     overlapping = [list(g) for g in disjoint]
#     for k in range(K):
#         in_group = set(disjoint[k])
#         for j in range(conf_mat.shape[0]):
#             if j in in_group:
#                 continue
#             if conf_mat[j, list(in_group)].sum() >= u_t:
#                 overlapping[k].append(j)

#     return [sorted(set(g)) for g in overlapping]

# conf_mat = build_confusion_matrix(model, val_loader)
# coarse_groups = build_hierarchy(conf_mat, K=9, gamma=5)
# for k, g in enumerate(coarse_groups):
#     print(f"Coarse group {k}: {len(g)} classes")

In [ ]:
# def init_from_baseline(hd_model, baseline):
#     sd = baseline.state_dict()

#     # Shared backbone: map baseline layer names to Sequential indices
#     # stem = [conv1(0), bn1(1), relu(2), maxpool(3), layer1(4), layer2(5)]
#     name_to_idx = {'conv1': 0, 'bn1': 1, 'layer1': 4, 'layer2': 5}
#     shared_sd = {}
#     for name, idx in name_to_idx.items():
#         for k, v in {k2: v2 for k2, v2 in sd.items() if k2.startswith(name + '.')}.items():
#             shared_sd[f'stem.{idx}.{k[len(name)+1:]}'] = v
#     hd_model.shared.load_state_dict(shared_sd, strict=False)

#     # Coarse component: layer3, layer4, fc
#     coarse_sd = {}
#     for i, name in enumerate(['layer3', 'layer4']):
#         for k, v in {k2: v2 for k2, v2 in sd.items() if k2.startswith(name + '.')}.items():
#             coarse_sd[f'backend.{i}.{k[len(name)+1:]}'] = v
#     for k, v in {k2: v2 for k2, v2 in sd.items() if k2.startswith('fc.')}.items():
#         coarse_sd[f'fc.{k[3:]}'] = v
#     hd_model.coarse.load_state_dict(coarse_sd, strict=False)

#     # Seed all fine component backends from baseline layer3/layer4
#     for fine_comp in hd_model.fine_components:
#         fine_sd = {}
#         for i, name in enumerate(['layer3', 'layer4']):
#             for k, v in {k2: v2 for k2, v2 in sd.items() if k2.startswith(name + '.')}.items():
#                 fine_sd[f'backend.{i}.{k[len(name)+1:]}'] = v
#         fine_comp.load_state_dict(fine_sd, strict=False)

#     print("Weights initialized from baseline.")

# hd_model = HDCNN(coarse_groups, num_fine=100, pretrained_backbone=False).to(DEVICE)
# init_from_baseline(hd_model, model)

In [ ]:
# hdcnn_history = {"train_loss": [], "val_loss": [], "train_accs": [], "val_accs": []}
# for epoch in range(EPOCHS):
#     train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
#     val_loss, val_acc = evaluate(model, val_loader, criterion)

#     history["train_loss"].append(train_loss)
#     history["val_loss"].append(val_loss)
#     history["train_accs"].append(train_acc)
#     history["val_accs"].append(val_acc)
#     print(f"  Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.3f} | Val: {val_acc:.3f}")


# torch.save(model.state_dict(), "resnet18_cifar100.pth")
# print("\nDone. Model saved to resnet18_cifar100.pth")